# Numerical Computation for Deep Learning — PyTorch Edition

A teaching notebook on **numerical stability in deep learning**, developed with reference to the slides *04_numerical.pptx* and the structure of the `numerical_errors.ipynb` notebook.

## Learning objectives

By the end of this notebook, you should be able to:

- distinguish **precision** from the **dynamic range** of floating-point types;
- recognize overflow, underflow, loss of information, and catastrophic cancellation;
- understand why mathematically equivalent formulas may behave differently on a computer;
- use numerically stable PyTorch primitives (`logsumexp`, `log_softmax`, `cross_entropy`, `BCEWithLogitsLoss`, ...);
- diagnose `NaN`, `inf`, and problematic gradients;
- understand why FP16/BF16 and mixed precision require specific numerical care.

> **Guiding idea:** in deep learning, it is not enough for a formula to be mathematically correct. It must also be implemented stably with a finite number of bits.

## Slides → PyTorch map

| Topic | Naive version / risk | Recommended PyTorch approach |
|---|---|---|
| `log(sum(exp(x)))` | overflow / underflow | `torch.logsumexp` |
| `log(softmax(x))` | saturation and `log(0)` | `F.log_softmax` |
| multiclass cross-entropy | softmax → log → mean | `F.cross_entropy` |
| binary cross-entropy | sigmoid → BCE | `F.binary_cross_entropy_with_logits` / `nn.BCEWithLogitsLoss` |
| normalization | division by an almost-zero standard deviation | `sqrt(var + eps)` in normalization layers |
| mixed precision | FP16 gradient underflow | `torch.autocast` + `torch.amp.GradScaler` |

The main sequence follows the part of the slides devoted to: numerical precision → rounding → overflow/underflow → subtraction → `log`/`sqrt` → log-sum-exp → softmax → cross-entropy → bug hunting.

## 0. Setup

In [ ]:
import math
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

torch.manual_seed(0)

## 1. Floating point: precision and dynamic range

A real number is represented using a finite number of bits. Two properties are especially important:

- **precision**: how finely we can distinguish nearby numbers;
- **dynamic range**: how small or large representable numbers can be.

`torch.finfo` lets us inspect the numerical properties of a dtype directly.

In [ ]:
dtypes = [torch.float16, torch.bfloat16, torch.float32, torch.float64]

for dtype in dtypes:
    f = torch.finfo(dtype)
    print(
        f"{str(dtype):15s} "
        f"bits={f.bits:2d}  "
        f"eps={f.eps:.3e}  "
        f"tiny={f.tiny:.3e}  "
        f"max={f.max:.3e}"
    )

### Observation

`eps` is the distance between 1 and the next representable number greater than 1. Therefore, there is no uniform “absolute precision” over the entire real line: the spacing between representable numbers grows with their magnitude.

**Question:** why does BF16 have a worse `eps` than FP16, while having a dramatically larger `max`?

## 2. Rounding: adding a small number may have no effect

This directly reproduces the example from the slides: in `float32`, `1 + 1e-8` may be rounded exactly to `1`.

In [ ]:
a = torch.tensor([0.0, 1e-8], dtype=torch.float32)

print("a              =", a)
print("a + 1          =", a + 1.0)
print("(a + 1) - 1   =", (a + 1.0) - 1.0)

The last line is the key one: after rounding, the information carried by the small term is **not recoverable**.

In [ ]:
for dtype in [torch.float16, torch.bfloat16, torch.float32, torch.float64]:
    x = torch.tensor(1e-8, dtype=dtype)
    one = torch.tensor(1.0, dtype=dtype)
    print(dtype, "x =", x.item(), "  1+x =", (one + x).item())

## 3. Floating-point arithmetic is not associative

Over the real numbers:

\[
(a+b)+c = a+(b+c)
\]

In floating point, this is not necessarily true.

In [ ]:
a = torch.tensor(1e8, dtype=torch.float32)
b = torch.tensor(-1e8, dtype=torch.float32)
c = torch.tensor(1.0, dtype=torch.float32)

left = (a + b) + c
right = a + (b + c)

print("(a+b)+c =", left.item())
print("a+(b+c) =", right.item())

This explains why reductions, parallel execution, and different backends can produce small differences even with identical inputs.

### Mini-experiment: summation order

In [ ]:
x = torch.cat([
    torch.tensor([1e8], dtype=torch.float32),
    torch.ones(100_000, dtype=torch.float32),
    torch.tensor([-1e8], dtype=torch.float32),
])

print("torch.sum float32:", x.sum().item())
print("sum after float64 cast:", x.double().sum().item())

## 4. Overflow and underflow of `exp`

The slides point out that `exp(x)` overflows at around 89 in float32. Instead of memorizing that number, we can derive the threshold from the dtype:

\[
\exp(x) \leq \text{finfo.max} \quad\Rightarrow\quad x \lesssim \log(\text{finfo.max})
\]

In [ ]:
for dtype in [torch.float16, torch.bfloat16, torch.float32, torch.float64]:
    f = torch.finfo(dtype)
    print(dtype, "log(max) ≈", math.log(f.max))

In [ ]:
values = torch.tensor([10.0, 12.0, 80.0, 89.0, 100.0])

for dtype in [torch.float16, torch.float32, torch.float64]:
    print("\n", dtype)
    print(torch.exp(values.to(dtype)))

### Underflow

For very negative inputs, `exp(x)` becomes so small that it is rounded to zero.

In [ ]:
values = torch.tensor([-10.0, -50.0, -100.0, -1000.0])

for dtype in [torch.float16, torch.float32, torch.float64]:
    print("\n", dtype)
    print(torch.exp(values.to(dtype)))

## 5. Secondary effects: from `inf` to `NaN`

An overflow does not necessarily remain confined to the operation that produced it.

In [ ]:
x = torch.exp(torch.tensor(100.0, dtype=torch.float32))
y = torch.exp(torch.tensor(100.0, dtype=torch.float32))
z = x - y

print("x =", x)
print("y =", y)
print("x-y =", z)
print("isfinite?", torch.isfinite(z).item())

`inf - inf` has no well-defined value and produces `NaN`. This is a common pattern: the operation that produces the `NaN` may occur **much later** than the operation that caused the original overflow.

## 6. Catastrophic cancellation: the variance example

A mathematically valid formula for the variance is

\[
\operatorname{Var}(X) = E[X^2] - E[X]^2.
\]

If the two terms are large and nearly equal, the subtraction can cancel many significant digits.

In [ ]:
x = 1000.0 + 0.1 * torch.arange(10, dtype=torch.float32)

var_naive = (x * x).mean() - x.mean() ** 2
var_torch = torch.var(x, correction=0)
var_centered = ((x - x.mean()) ** 2).mean()

print("naive E[x²]-E[x]² =", var_naive)
print("centered form       =", var_centered)
print("torch.var           =", var_torch)

In [ ]:
print("sqrt(naive variance) =", torch.sqrt(var_naive))
print("sqrt(torch variance) =", torch.sqrt(var_torch))

This example connects three issues from the slides:

1. subtraction between quantities of similar magnitude;
2. a variance that can become numerically incorrect;
3. `sqrt` of a negative number → `NaN`.

### Comparison with float64

In [ ]:
x64 = x.double()
var_naive64 = (x64 * x64).mean() - x64.mean() ** 2
print("naive float64 =", var_naive64)

## 7. `log` and `sqrt`: domains and problematic derivatives

- `log(0) = -inf`;
- `log(x<0)` produces `NaN` over the reals;
- `sqrt(0)=0`, but the derivative \(1/(2\sqrt{x})\) diverges at zero.

In [ ]:
vals = torch.tensor([1.0, 0.0, -1.0], dtype=torch.float32)
print("log :", torch.log(vals))
print("sqrt:", torch.sqrt(vals))

### Autograd makes the derivative problem of `sqrt` visible

In [ ]:
x = torch.tensor(0.0, requires_grad=True)
y = torch.sqrt(x)
y.backward()

print("sqrt(0) =", y.item())
print("gradient =", x.grad)

In [ ]:
x = torch.tensor(0.0, requires_grad=True)
eps = 1e-6
y = torch.sqrt(x + eps)
y.backward()

print("sqrt(eps) =", y.item())
print("gradient   =", x.grad.item())

## 8. Where should `eps` go?

The slides ask whether we should use

```python
sqrt(variance + eps)
```

or

```python
sqrt(variance) + eps
```

The two expressions are **not equivalent**. Normalization layers typically use the first form, with `eps` inside the square root.

In [ ]:
variance = torch.tensor([0.0, 1e-12, 1e-8, 1e-4, 1.0])
eps = 1e-5

inside = torch.sqrt(variance + eps)
outside = torch.sqrt(variance) + eps

print(torch.stack([variance, inside, outside], dim=1))

When the variance is small, the position of `eps` changes the denominator substantially and therefore changes the function being implemented.

## 9. Specialized functions: `log1p` and `expm1`

For small \(x\), computing `1+x` first may completely lose the increment. This is why specialized primitives exist:

- `torch.log1p(x)` computes \(\log(1+x)\);
- `torch.expm1(x)` computes \(e^x-1\).

In [ ]:
x = torch.tensor(1e-8, dtype=torch.float32)

print("torch.log(1+x) =", torch.log(1 + x).item())
print("torch.log1p(x)  =", torch.log1p(x).item())
print()
print("torch.exp(x)-1  =", (torch.exp(x) - 1).item())
print("torch.expm1(x)  =", torch.expm1(x).item())

### Visualizing the error in `log(1+x)`

In [ ]:
xs = torch.logspace(-12, -2, 200, dtype=torch.float32)
naive = torch.log(1 + xs)
stable = torch.log1p(xs)
reference = torch.log1p(xs.double()).float()

err_naive = (naive - reference).abs()
err_stable = (stable - reference).abs()

plt.figure(figsize=(7, 4))
plt.loglog(xs.numpy(), (err_naive + 1e-30).numpy(), label="log(1+x)")
plt.loglog(xs.numpy(), (err_stable + 1e-30).numpy(), label="log1p(x)")
plt.xlabel("x")
plt.ylabel("absolute error")
plt.title("Small x: naive vs specialized implementation")
plt.legend()
plt.grid(True, which="both", alpha=0.25)
plt.show()

## 10. `log(exp(x))` and Softplus

The slides note that `log(exp(x))` should be simplified to `x`. The reason is not only efficiency: the intermediate `exp(x)` may overflow.

In [ ]:
for value in [1.0, 50.0, 100.0, -100.0]:
    x = torch.tensor(value, dtype=torch.float32)
    naive = torch.log(torch.exp(x))
    print(f"x={value:7.1f}  log(exp(x))={naive.item():>12}  stable={x.item():>8}")

### Softplus

\[
\operatorname{softplus}(x)=\log(1+e^x)
\]

This is a perfect example of a simple formula that requires a stable implementation.

In [ ]:
x = torch.tensor([10.0, 50.0, 100.0], dtype=torch.float32)

naive = torch.log(1 + torch.exp(x))
stable = F.softplus(x)

print("naive :", naive)
print("stable:", stable)

# Part II — Stability of probability functions and losses

This is the most important part for practical deep learning: many loss functions combine `exp`, normalization, and `log`.

## 11. LogSumExp

The naive expression

\[
\log \sum_i e^{x_i}
\]

fails if one element is very large (overflow) or if all elements are very negative (underflow).

The stable transformation uses \(m=\max_i x_i\):

\[
\log\sum_i e^{x_i} = m + \log\sum_i e^{x_i-m}.
\]

In [ ]:
def naive_logsumexp(x, dim=0):
    return torch.log(torch.exp(x).sum(dim=dim))


def stable_logsumexp_manual(x, dim=0):
    m = x.max(dim=dim, keepdim=True).values
    return (m + torch.log(torch.exp(x - m).sum(dim=dim, keepdim=True))).squeeze(dim)

for x in [
    torch.tensor([1000.0, 1000.0]),
    torch.tensor([-1000.0, -1000.0]),
]:
    print("x =", x.tolist())
    print("naive  =", naive_logsumexp(x))
    print("manual =", stable_logsumexp_manual(x))
    print("torch   =", torch.logsumexp(x, dim=0))
    print()

## 12. Naive vs stable Softmax

A literal implementation of

\[
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
\]

may produce `inf/inf`.

In [ ]:
def softmax_naive(x, dim=-1):
    e = torch.exp(x)
    return e / e.sum(dim=dim, keepdim=True)


def softmax_stable_manual(x, dim=-1):
    z = x - x.max(dim=dim, keepdim=True).values
    e = torch.exp(z)
    return e / e.sum(dim=dim, keepdim=True)

logits = torch.tensor([[1000.0, 999.0, 998.0]])

print("naive :", softmax_naive(logits, dim=1))
print("manual:", softmax_stable_manual(logits, dim=1))
print("F.softmax:", F.softmax(logits, dim=1))

## 13. `log(softmax(x))` vs `log_softmax(x)`

This is one of the most important replacements to learn in PyTorch.

In [ ]:
logits = torch.tensor([[1000.0, 0.0, -1000.0]])

bad = torch.log(F.softmax(logits, dim=1))
good = F.log_softmax(logits, dim=1)

print("log(softmax):", bad)
print("log_softmax :", good)

The probability of a very unlikely class may be rounded to zero; once that happens, `log(0)` is irreversibly `-inf`. `log_softmax` avoids materializing that extreme probability in the naive way.

## 14. Cross-entropy: use logits, not probabilities

The slides emphasize that cross-entropy should be computed directly from logits.

### Naive implementation

In [ ]:
logits = torch.tensor([[1000.0, 0.0, -1000.0]], requires_grad=True)
target = torch.tensor([1])

p = F.softmax(logits, dim=1)
loss_bad = -torch.log(p[0, target[0]])

print("probabilities =", p)
print("loss_bad      =", loss_bad)

loss_bad.backward()
print("gradient bad  =", logits.grad)

### Correct implementation with `F.cross_entropy`

In [ ]:
logits = torch.tensor([[1000.0, 0.0, -1000.0]], requires_grad=True)
target = torch.tensor([1])

loss_good = F.cross_entropy(logits, target)
print("loss_good     =", loss_good)

loss_good.backward()
print("gradient good =", logits.grad)

The crucial point is not only to obtain a finite loss: **we need a usable gradient**.

## 15. Binary case: Sigmoid + BCE vs BCEWithLogits

The same principle applies to binary classification.

In [ ]:
z = torch.tensor([100.0], requires_grad=True)
y = torch.tensor([0.0])

p = torch.sigmoid(z)
loss_bad = -(y * torch.log(p) + (1-y) * torch.log(1-p)).mean()

print("sigmoid(z) =", p)
print("manual BCE =", loss_bad)

loss_bad.backward()
print("bad grad   =", z.grad)

In [ ]:
z = torch.tensor([100.0], requires_grad=True)
y = torch.tensor([0.0])

loss_good = F.binary_cross_entropy_with_logits(z, y)
print("BCEWithLogits =", loss_good)

loss_good.backward()
print("good grad     =", z.grad)

**Practical rule:**

- multiclass → `F.cross_entropy(logits, target)`;
- binary/multilabel → `F.binary_cross_entropy_with_logits(logits, target)` or `nn.BCEWithLogitsLoss`.

## 16. Modern example: attention and extreme softmax values

Scaled dot-product attention contains a softmax:

\[
\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V.
\]

With very large scores, a naively implemented softmax can fail.

In [ ]:
Q = torch.tensor([[1000.0, 1000.0], [1000.0, -1000.0]])
K = torch.tensor([[1000.0, 1000.0], [-1000.0, 1000.0]])
V = torch.tensor([[1.0, 0.0], [0.0, 1.0]])

scores = Q @ K.T / math.sqrt(Q.shape[-1])

weights_bad = softmax_naive(scores, dim=-1)
weights_good = F.softmax(scores, dim=-1)

print("scores:\n", scores)
print("\nnaive attention weights:\n", weights_bad)
print("\nstable attention weights:\n", weights_good)
print("\nstable attention output:\n", weights_good @ V)

This directly connects numerical stability to Transformers: a stable softmax is not a low-level implementation detail, but a necessary condition for fundamental blocks of modern architectures to work.

# Part III — Reduced precision and mixed precision

## 17. FP16 vs BF16 vs FP32

FP16 and BF16 both use fewer bits than FP32, but in different ways:

- FP16 has greater mantissa precision than BF16, but a much smaller dynamic range;
- BF16 retains a range similar to FP32, but has coarser precision.

In [ ]:
values = [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]

for v in values:
    row = []
    for dtype in [torch.float16, torch.bfloat16, torch.float32]:
        row.append(torch.tensor(v, dtype=dtype).item())
    print(f"{v:>8.1e}   fp16={row[0]:>12.4e}   bf16={row[1]:>12.4e}   fp32={row[2]:>12.4e}")

### Example of a gradient that underflows in FP16

In [ ]:
g = torch.tensor([1e-8], dtype=torch.float32)
print("float32:", g)
print("float16:", g.to(torch.float16))
print("bfloat16:", g.to(torch.bfloat16))

This is the conceptual reason for **gradient scaling**: temporarily multiplying the loss shifts gradients toward magnitudes that are more representable during the backward pass.

## 18. Automatic Mixed Precision (AMP)

On a CUDA GPU, the modern PyTorch pattern is:

1. use `torch.autocast` for the forward pass and loss computation;
2. use `torch.amp.GradScaler` to reduce the risk of FP16 gradient underflow;
3. run backward outside the `autocast` context.

The following cell can be executed in Colab with a GPU runtime; on CPU it is skipped.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    model = nn.Sequential(
        nn.Linear(32, 64),
        nn.ReLU(),
        nn.Linear(64, 10),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler("cuda")

    x = torch.randn(128, 32, device=device)
    target = torch.randint(0, 10, (128,), device=device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(x)
        loss = F.cross_entropy(logits, target)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    print("AMP step completed. loss =", loss.item())
else:
    print("CUDA not available: AMP cell skipped. In Colab: Runtime → Change runtime type → GPU.")

## 19. The final result may be representable while intermediate values are not

A common mistake is to think: “if the result lies within the float32 range, then the computation is safe.”

The Euclidean norm contains intermediate squares. For very large components, those squares may overflow even though the final norm itself would be representable.

In [ ]:
a = torch.tensor([1e20, 1e20], dtype=torch.float32)

print("float32 norm:", a.norm())
print("float64 norm:", a.double().norm())

# Part IV — Linear algebra and conditioning

## 20. Nearly singular matrices

Stability depends not only on the dtype, but also on the **conditioning of the problem**. A nearly singular matrix amplifies small errors in the data and in floating-point rounding.

In [ ]:
A64 = torch.tensor([
    [1.0, 1.0],
    [1.0, 1.0 + 1e-7],
], dtype=torch.float64)
b64 = torch.tensor([2.0, 2.0 + 1e-7], dtype=torch.float64)

A32 = A64.float()
b32 = b64.float()

print("cond(A64) =", torch.linalg.cond(A64).item())
print("solution float64 =", torch.linalg.solve(A64, b64))

try:
    print("solution float32 =", torch.linalg.solve(A32, b32))
except RuntimeError as e:
    print("float32 solve failed:", e)

### A small perturbation of the data

In [ ]:
A = torch.tensor([
    [1.0, 1.0],
    [1.0, 1.0 + 1e-10],
], dtype=torch.float64)

b1 = torch.tensor([2.0, 2.0 + 1e-10], dtype=torch.float64)
b2 = b1.clone()
b2[1] += 1e-12

x1 = torch.linalg.solve(A, b1)
x2 = torch.linalg.solve(A, b2)

print("x1 =", x1)
print("x2 =", x2)
print("delta solution =", x2 - x1)
print("cond(A) =", torch.linalg.cond(A).item())

# Part V — Numerical debugging in PyTorch

## 21. Systematically check for `NaN` and `inf`

In [ ]:
def tensor_report(name, x):
    x_detached = x.detach()
    finite = torch.isfinite(x_detached)
    print(f"{name}")
    print("  shape    :", tuple(x_detached.shape))
    print("  dtype    :", x_detached.dtype)
    print("  allfinite:", finite.all().item())
    print("  nan      :", torch.isnan(x_detached).sum().item())
    print("  inf      :", torch.isinf(x_detached).sum().item())
    if finite.any():
        xf = x_detached[finite]
        print("  min/max  :", xf.min().item(), xf.max().item())

x = torch.tensor([1.0, float("inf"), float("nan")])
tensor_report("example", x)

## 22. Check gradients during training

In [ ]:
def report_nonfinite_gradients(model):
    bad = []
    for name, p in model.named_parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            bad.append(name)
    return bad

model = nn.Linear(4, 2)
x = torch.randn(8, 4)
y = torch.randint(0, 2, (8,))

loss = F.cross_entropy(model(x), y)
loss.backward()

print("non-finite gradients:", report_nonfinite_gradients(model))

## 23. `detect_anomaly`: locating problematic operations in backward

`torch.autograd.detect_anomaly()` is very useful for debugging, but it is expensive: it should not normally remain enabled during training.

In [ ]:
x = torch.tensor(0.0, requires_grad=True)

try:
    with torch.autograd.detect_anomaly():
        # 0/0 produces NaN in the forward pass and a problematic backward pass
        y = x / x
        y.backward()
except RuntimeError as e:
    print("Anomaly detected:")
    print(str(e).splitlines()[0])

## 24. `nan_to_num` is not a cure

`torch.nan_to_num` can be useful for sanitizing data or outputs in controlled situations, but it **must not hide a numerically unstable loss or gradient**.

In [ ]:
x = torch.tensor([1.0, float("nan"), float("inf"), -float("inf")])
print(torch.nan_to_num(x))

If a `NaN` appears during training, the priority is to find **the first operation that generates a non-finite value**, rather than replacing it at the end of the pipeline.

# Part VI — Mini-lab: unstable vs stable pipeline

Let us combine the previous ideas in a single multiclass example.

In [ ]:
torch.manual_seed(1)

# Simulate very large logits: a situation that may arise from exploding weights/activations.
logits = 200.0 * torch.randn(16, 5, requires_grad=True)
target = torch.randint(0, 5, (16,))

# Naive pipeline
p = softmax_naive(logits, dim=1)
loss_bad = -torch.log(p[torch.arange(len(target)), target]).mean()

print("bad loss:", loss_bad)
print("bad loss finite?", torch.isfinite(loss_bad).item())

In [ ]:
# Stable pipeline
logits2 = logits.detach().clone().requires_grad_(True)
loss_good = F.cross_entropy(logits2, target)
loss_good.backward()

print("good loss:", loss_good)
print("good loss finite?", torch.isfinite(loss_good).item())
print("good gradients finite?", torch.isfinite(logits2.grad).all().item())

# Practical checklist

When you encounter `NaN`, `inf`, a stuck loss, or suspicious gradients:

1. **Check the dtype** (`float16`, `bfloat16`, `float32`, `float64`).
2. Look for `exp`, `log`, `sqrt`, divisions, and subtractions between similar numbers.
3. Inspect intermediate values with `torch.isfinite`.
4. Use loss functions that operate directly on **logits**.
5. Prefer `torch.logsumexp`, `F.log_softmax`, `F.cross_entropy`, and `BCEWithLogitsLoss` over manual compositions.
6. Check whether normalization uses an appropriate `eps`.
7. With FP16, consider underflow/overflow and use AMP + gradient scaling.
8. For linear algebra problems, inspect the **condition number**.
9. Use `detect_anomaly()` temporarily to locate a problematic backward pass.
10. Do not use `nan_to_num` to mask a numerically unstable pipeline.

# Exercises

### Exercise 1 — overflow threshold
Write a function that, given a `torch.dtype`, estimates the largest `x` for which `exp(x)` remains finite. Verify the result experimentally.

### Exercise 2 — summation
Build a vector containing both very large and very small numbers. Compare `sum` in float32 and float64, and try different orderings of the elements.

### Exercise 3 — variance
Generate data of the form `offset + noise`, progressively increasing `offset`. Compare:

```python
(x*x).mean() - x.mean()**2
```

with `torch.var`.

### Exercise 4 — cross-entropy
Progressively increase the scale of the logits from 1 to 1000 and determine when the `softmax -> log` pipeline starts producing non-finite values.

### Exercise 5 — mixed precision
On a GPU, compare a small training loop in FP32 and with AMP. Record the loss, runtime, and whether non-finite gradients occur.

### Exercise 6 — attention
Increase the norm of `Q` and `K` and compare the naive softmax with `F.softmax`.

# References

- Course slides: *04_numerical.pptx*, section **Numerical Precision: A deep learning super skill** and the following sections.
- Starting notebook indicated by the instructor: `serivan/DeepLearning/02-Preliminaries/numerical_errors.ipynb`.
- PyTorch documentation: *Numerical accuracy*, `torch.finfo`, `torch.logsumexp`, `F.log_softmax`, `BCEWithLogitsLoss`, *Automatic Mixed Precision*.

> Note: this version is a complete PyTorch reconstruction aligned with the contents of the slides. It is not a mechanical cell-by-cell conversion of the remote notebook.